# Import Library

In [1]:
import pandas as pd
import sqlite3
import os

# Load Dataset

In [6]:
# Cek posisi working directory sekarang
print(f"📍 Current directory: {os.getcwd()}")

# Cek isi folder
print(f"\n📁 Isi folder saat ini:")
for item in os.listdir('.'):
    print(f"   {item}")

📍 Current directory: d:\Project Portofolio\Data Analyst, Scientiest, ML, DL, AI\Retail Shrinkage & Loss Prevention Analytics\retail_shrinkage_analytics

📁 Isi folder saat ini:
   .git
   .gitignore
   data
   notebook
   outputs
   raw_data
   Readme.md
   requirements.txt
   sql
   venv


In [7]:
products = pd.read_csv('raw_data/product.csv')
df = pd.read_csv('data/transactions_with_shrinkage.csv')


In [8]:
# Buat SQLite database (file lokal, tidak perlu install server)
conn = sqlite3.connect('data/retail_shrinkage.db')

In [9]:
# Load ke database
df.to_sql('transactions', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

print("✅ Database created!")
print(f"   Tables: transactions, products")

✅ Database created!
   Tables: transactions, products


In [10]:
# Helper function — kita pakai ini terus
def query(sql):
    return pd.read_sql_query(sql, conn)

print("\n📋 Verifikasi tabel:")
print(query("SELECT COUNT(*) as total_rows FROM transactions"))


📋 Verifikasi tabel:
   total_rows
0     2597132


# SQL Analysis

## Kategori Produk Paling Rentan Shrinkage

In [11]:
q1 = query("""
    SELECT 
        p.DEPARTMENT,
        p.COMMODITY_DESC,
        COUNT(*)                                    AS total_transactions,
        SUM(CASE WHEN t.IS_FRAUD = 1 THEN 1 
                ELSE 0 END)                        AS fraud_count,
        ROUND(
            SUM(CASE WHEN t.IS_FRAUD = 1 THEN 1 
                     ELSE 0 END) * 100.0 / COUNT(*), 4
        )                                           AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN t.IS_FRAUD = 1 
                    THEN ABS(t.SALES_VALUE) 
                    ELSE 0 END), 2)              AS estimated_loss
    FROM transactions t
    LEFT JOIN products p ON t.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY p.DEPARTMENT, p.COMMODITY_DESC
    HAVING fraud_count > 0
    ORDER BY estimated_loss DESC
    LIMIT 20
""")

print("TOP 20 KATEGORI PRODUK PALING RENTAN SHRINKAGE")
print("=" * 65)
print(q1.to_string(index=False))

TOP 20 KATEGORI PRODUK PALING RENTAN SHRINKAGE
DEPARTMENT                 COMMODITY_DESC  total_transactions  fraud_count  fraud_rate_pct  estimated_loss
   DRUG GM               CANDY - PACKAGED               34195           25          0.0731          267.80
   DRUG GM                       MAGAZINE                9795           20          0.2042          254.08
   DRUG GM             HAIR CARE PRODUCTS               12183           26          0.2134          223.53
   GROCERY                       HISPANIC               24003           19          0.0792          223.14
 COSMETICS           MAKEUP AND TREATMENT                6014           28          0.4656          214.43
   DRUG GM GREETING CARDS/WRAP/PARTY SPLY               11938           18          0.1508          202.57
   GROCERY                    COLD CEREAL               37885           15          0.0396          193.56
   GROCERY         FRZN MEAT/MEAT DINNERS               56078           14          0.0250       

### 💡 Insight Bisnis:

**Pattern 1: DRUG GM dominasi top losses**
```
DRUG GM muncul 7x di top 20
→ Departemen ini punya produk bernilai tinggi
  (obat, kosmetik, majalah) yang mudah di-pocket
→ Prioritas pemasangan security tag!
```

**Pattern 2: Fraud rate vs Estimated loss — bedanya penting!**
```
BOOKSTORE    → fraud_rate 0.8985% (TERTINGGI!)
CANDY        → fraud_rate 0.0731% (rendah)
tapi CANDY   → estimated_loss $267 > BOOKSTORE $147

Artinya:
BOOKSTORE  = jarang terjadi tapi hampir selalu fraud kalau ada transaksi
CANDY      = sering terjadi, fraud-nya kecil per kejadian tapi volume tinggi

## Pola Waktu Anomali Transaksi

In [12]:
q2 = query("""
    WITH time_categorized AS (
        SELECT 
            IS_FRAUD,
            SHRINKAGE_TYPE,
            SALES_VALUE,
            CASE 
                WHEN TRANS_TIME BETWEEN 0    AND 559  THEN '1. Midnight (00-06)'
                WHEN TRANS_TIME BETWEEN 600  AND 859  THEN '2. Early Morning (06-09)'
                WHEN TRANS_TIME BETWEEN 900  AND 1159 THEN '3. Morning (09-12)'
                WHEN TRANS_TIME BETWEEN 1200 AND 1459 THEN '4. Afternoon (12-15)'
                WHEN TRANS_TIME BETWEEN 1500 AND 1759 THEN '5. Late Afternoon (15-18)'
                WHEN TRANS_TIME BETWEEN 1800 AND 2059 THEN '6. Evening (18-21)'
                ELSE                                       '7. Night (21-24)'
            END AS time_slot
        FROM transactions
    )
    SELECT
        time_slot,
        COUNT(*)                                        AS total_transactions,
        SUM(CASE WHEN IS_FRAUD = 1 THEN 1 ELSE 0 END)  AS fraud_count,
        ROUND(
            SUM(CASE WHEN IS_FRAUD = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4
        )                                               AS fraud_rate_pct,
        ROUND(AVG(CASE WHEN IS_FRAUD = 1 
                    THEN ABS(SALES_VALUE) END), 2)   AS avg_fraud_value
    FROM time_categorized
    GROUP BY time_slot
    ORDER BY time_slot
""")

print("⏰ POLA WAKTU ANOMALI TRANSAKSI")
print("=" * 70)
print(q2.to_string(index=False))

⏰ POLA WAKTU ANOMALI TRANSAKSI
                time_slot  total_transactions  fraud_count  fraud_rate_pct  avg_fraud_value
      1. Midnight (00-06)               41339          146          0.3532            26.46
 2. Early Morning (06-09)               59052          371          0.6283             2.42
       3. Morning (09-12)              337165          232          0.0688             6.86
     4. Afternoon (12-15)              593064          289          0.0487             7.51
5. Late Afternoon (15-18)              728906           55          0.0075             0.00
       6. Evening (18-21)              627587          135          0.0215             0.00
         7. Night (21-24)              210019          172          0.0819             7.49


### 🚨 Dua Temuan Paling Kritis:

**Temuan 1: Midnight punya avg_fraud_value TERTINGGI**
```
Midnight (00-06)  → avg fraud value $26.46  ← 3-4x lebih tinggi!
Morning  (09-12)  → avg fraud value  $6.86
Afternoon(12-15)  → avg fraud value  $7.51
```

> Ini konfirmasi **Sweep Fraud** kita — transaksi tengah malam nilainya jauh lebih besar. Sedikit kejadian tapi **damage per incident sangat tinggi**.

**Temuan 2: Early Morning punya fraud_rate TERTINGGI**
```
Early Morning (06-09) → fraud_rate 0.6283%  ← tertinggi!
Midnight      (00-06) → fraud_rate 0.3532%
Normal hours          → fraud_rate < 0.09%
```

> Jam 06-09 adalah **shift pergantian** — pengawasan longgar, kamera mungkin belum aktif semua, supervisor baru datang.

### 📊 Pattern Lengkap:
```
JAM OPERASIONAL NORMAL (09-21):
fraud_rate → 0.007% - 0.082%  ← rendah, pengawasan ketat

JAM TIDAK NORMAL (00-09):
fraud_rate → 0.35% - 0.63%    ← 4-8x lebih tinggi!

## Store dengan Void Rate Abnormal

In [13]:
q3 = query("""
    WITH store_stats AS (
        SELECT
            STORE_ID,
            COUNT(*)                                        AS total_transactions,
            SUM(CASE WHEN IS_FRAUD = 1 THEN 1 ELSE 0 END)  AS fraud_count,
            SUM(CASE WHEN SALES_VALUE < 0 THEN 1 ELSE 0 END) AS void_count,
            SUM(CASE WHEN SALES_VALUE = 0 
                    AND IS_FRAUD = 1 THEN 1 ELSE 0 END)    AS zero_fraud_count,
            ROUND(ABS(SUM(CASE WHEN IS_FRAUD = 1 
                        THEN SALES_VALUE ELSE 0 END)), 2)  AS total_loss
        FROM transactions
        GROUP BY STORE_ID
    ),
    -- Hitung rata-rata dan standar deviasi untuk threshold
    global_stats AS (
        SELECT
            AVG(fraud_count * 100.0 / total_transactions)   AS avg_fraud_rate,
            AVG(fraud_count * 100.0 / total_transactions) + 
            (2 * AVG(fraud_count * 100.0 / total_transactions)) AS threshold_yellow,
            AVG(fraud_count * 100.0 / total_transactions) + 
            (4 * AVG(fraud_count * 100.0 / total_transactions)) AS threshold_red
        FROM store_stats
    )
    SELECT
        s.STORE_ID,
        s.total_transactions,
        s.fraud_count,
        s.void_count,
        ROUND(s.fraud_count * 100.0 / s.total_transactions, 4) AS fraud_rate_pct,
        s.total_loss,
        CASE
            WHEN s.fraud_count * 100.0 / s.total_transactions 
                >= g.threshold_red    THEN '🔴 HIGH RISK'
            WHEN s.fraud_count * 100.0 / s.total_transactions 
                >= g.threshold_yellow THEN '🟡 MEDIUM RISK'
            ELSE                            '🟢 NORMAL'
        END AS risk_level
    FROM store_stats s
    CROSS JOIN global_stats g
    WHERE s.fraud_count > 0
    ORDER BY fraud_rate_pct DESC
    LIMIT 20
""")

print("🏪 STORE DENGAN ANOMALI RATE TERTINGGI")
print("=" * 80)
print(q3.to_string(index=False))

print("\n📊 Distribusi Risk Level:")
print(q3['risk_level'].value_counts())

🏪 STORE DENGAN ANOMALI RATE TERTINGGI
 STORE_ID  total_transactions  fraud_count  void_count  fraud_rate_pct  total_loss  risk_level
     3294                  52           51          50         98.0769      480.42 🔴 HIGH RISK
     3298                  55           52          50         94.5455      462.23 🔴 HIGH RISK
     2915                   6            5           0         83.3333        0.00 🔴 HIGH RISK
      213                  63           51          50         80.9524      457.83 🔴 HIGH RISK
      495                   5            4           0         80.0000       68.89 🔴 HIGH RISK
      610                   5            4           0         80.0000       12.83 🔴 HIGH RISK
      648                   5            4           0         80.0000       18.80 🔴 HIGH RISK
      765                   5            4           0         80.0000       29.64 🔴 HIGH RISK
      783                   5            4           0         80.0000       48.78 🔴 HIGH RISK
     1433   

* STORE_ID 2915 → total_transactions = 6, fraud_count = 5 → 83%
* STORE_ID 495  → total_transactions = 5, fraud_count = 4 → 80%

Store dengan hanya 5-6 transaksi total langsung masuk HIGH RISK karena 4 dari 5 transaksinya adalah fraud simulasi kita.

Ini namanya Small Sample Size Problem:

* Store A: 10,000 transaksi, 100 fraud → fraud_rate = 1%   → mungkin LOW RISK
* Store B:      5 transaksi,   4 fraud → fraud_rate = 80%  → HIGH RISK? 

Secara statistik, Store B tidak bisa diandalkan!

4 kejadian dari 5 bisa jadi kebetulan semata.

💡 Solusinya: Minimum Transaction Threshold

Di dunia nyata, Loss Prevention hanya flag store yang punya volume transaksi signifikan. Kita perlu tambahkan filter minimum transaksi.

In [14]:
q3_fixed = query("""
    WITH store_stats AS (
        SELECT
            STORE_ID,
            COUNT(*)                                          AS total_transactions,
            SUM(CASE WHEN IS_FRAUD = 1 THEN 1 ELSE 0 END)    AS fraud_count,
            SUM(CASE WHEN SALES_VALUE < 0 THEN 1 ELSE 0 END) AS void_count,
            SUM(CASE WHEN SALES_VALUE = 0 
                    AND IS_FRAUD = 1 THEN 1 ELSE 0 END)      AS zero_fraud_count,
            ROUND(ABS(SUM(CASE WHEN IS_FRAUD = 1 
                        THEN SALES_VALUE ELSE 0 END)), 2)    AS total_loss
        FROM transactions
        GROUP BY STORE_ID
    ),
    -- Hanya store dengan transaksi signifikan
    qualified_stores AS (
        SELECT * FROM store_stats
        WHERE total_transactions >= 1000  -- ← minimum threshold
    ),
    global_stats AS (
        SELECT
            AVG(fraud_count * 100.0 / total_transactions)       AS avg_fraud_rate,
            AVG(fraud_count * 100.0 / total_transactions) +
            (2 * AVG(fraud_count * 100.0 / total_transactions)) AS threshold_yellow,
            AVG(fraud_count * 100.0 / total_transactions) +
            (4 * AVG(fraud_count * 100.0 / total_transactions)) AS threshold_red
        FROM qualified_stores
    )
    SELECT
        s.STORE_ID,
        s.total_transactions,
        s.fraud_count,
        s.void_count,
        ROUND(s.fraud_count * 100.0 / s.total_transactions, 4) AS fraud_rate_pct,
        s.total_loss,
        CASE
            WHEN s.fraud_count * 100.0 / s.total_transactions 
                >= g.threshold_red    THEN '🔴 HIGH RISK'
            WHEN s.fraud_count * 100.0 / s.total_transactions 
                >= g.threshold_yellow THEN '🟡 MEDIUM RISK'
            ELSE                            '🟢 NORMAL'
        END AS risk_level
    FROM qualified_stores s
    CROSS JOIN global_stats g
    ORDER BY fraud_rate_pct DESC
    LIMIT 20
""")

print("🏪 STORE ANOMALI RATE — QUALIFIED STORES (min 1000 transaksi)")
print("=" * 80)
print(q3_fixed.to_string(index=False))

print("\n📊 Distribusi Risk Level:")
print(q3_fixed['risk_level'].value_counts())

🏪 STORE ANOMALI RATE — QUALIFIED STORES (min 1000 transaksi)
 STORE_ID  total_transactions  fraud_count  void_count  fraud_rate_pct  total_loss    risk_level
      293               16461           53          50          0.3220      423.15   🔴 HIGH RISK
     3001                1201            2           0          0.1665        0.00   🔴 HIGH RISK
      569                1057            1           0          0.0946        0.00   🔴 HIGH RISK
     3182                1140            1           0          0.0877        0.00   🔴 HIGH RISK
      673                2761            2           0          0.0724       34.38 🟡 MEDIUM RISK
      286                6358            4           0          0.0629        0.00 🟡 MEDIUM RISK
    34280                8318            4           0          0.0481       39.13 🟡 MEDIUM RISK
    34007                2337            1           0          0.0428        0.00      🟢 NORMAL
      415                6176            2           0          0.

## Estimasi Kerugian Tahunan

In [15]:
q4 = query("""
    WITH shrinkage_by_dept AS (
        SELECT
            p.DEPARTMENT,
            t.SHRINKAGE_TYPE,
            COUNT(*)                                         AS incident_count,
            ROUND(ABS(SUM(t.SALES_VALUE)), 2)               AS total_loss_observed,
            ROUND(AVG(ABS(t.SALES_VALUE)), 2)               AS avg_loss_per_incident
        FROM transactions t
        LEFT JOIN products p ON t.PRODUCT_ID = p.PRODUCT_ID
        WHERE t.IS_FRAUD = 1
        GROUP BY p.DEPARTMENT, t.SHRINKAGE_TYPE
    ),
    revenue_by_dept AS (
        SELECT
            p.DEPARTMENT,
            ROUND(SUM(t.SALES_VALUE), 2)                    AS total_revenue,
            COUNT(*)                                         AS total_transactions
        FROM transactions t
        LEFT JOIN products p ON t.PRODUCT_ID = p.PRODUCT_ID
        WHERE t.IS_FRAUD = 0
        GROUP BY p.DEPARTMENT
    )
    SELECT
        s.DEPARTMENT,
        s.SHRINKAGE_TYPE,
        s.incident_count,
        s.total_loss_observed,
        s.avg_loss_per_incident,
        r.total_revenue,
        -- Estimasi loss rate
        ROUND(s.total_loss_observed * 100.0 / 
            NULLIF(r.total_revenue, 0), 4)                AS loss_rate_pct,
        -- Proyeksi tahunan (dataset = 2 tahun, jadi × 0.5)
        ROUND(s.total_loss_observed * 0.5, 2)               AS estimated_annual_loss,
        CASE
            WHEN s.total_loss_observed * 100.0 / 
                NULLIF(r.total_revenue, 0) > 0.5  THEN '🔴 CRITICAL'
            WHEN s.total_loss_observed * 100.0 / 
                NULLIF(r.total_revenue, 0) > 0.1  THEN '🟡 WARNING'
            ELSE                                        '🟢 MONITOR'
        END                                                  AS priority
    FROM shrinkage_by_dept s
    LEFT JOIN revenue_by_dept r ON s.DEPARTMENT = r.DEPARTMENT
    ORDER BY estimated_annual_loss DESC
    LIMIT 20
""")

print("💰 ESTIMASI KERUGIAN TAHUNAN PER KATEGORI & TIPE SHRINKAGE")
print("=" * 90)
print(q4.to_string(index=False))

# Summary total
print("\n📊 TOTAL ESTIMASI KERUGIAN TAHUNAN:")
print(f"   ${q4['estimated_annual_loss'].sum():,.2f}")

print("\n🔴 CRITICAL PRIORITY:")
print(q4[q4['priority'] == '🔴 CRITICAL'][
    ['DEPARTMENT','SHRINKAGE_TYPE','estimated_annual_loss']
].to_string(index=False))

💰 ESTIMASI KERUGIAN TAHUNAN PER KATEGORI & TIPE SHRINKAGE
DEPARTMENT SHRINKAGE_TYPE  incident_count  total_loss_observed  avg_loss_per_incident  total_revenue  loss_rate_pct  estimated_annual_loss   priority
   GROCERY    SWEEP_FRAUD              94              2424.23                  25.79     4093814.14         0.0592                1212.12  🟢 MONITOR
   DRUG GM     VOID_ABUSE             189              1843.04                   9.75     1055358.03         0.1746                 921.52  🟡 WARNING
   DRUG GM    SWEEP_FRAUD              67              1740.18                  25.97     1055358.03         0.1649                 870.09  🟡 WARNING
   GROCERY     VOID_ABUSE             182              1612.86                   8.86     4093814.14         0.0394                 806.43  🟢 MONITOR
 COSMETICS     VOID_ABUSE              25               224.30                   8.97       32360.37         0.6931                 112.15 🔴 CRITICAL
 NUTRITION    SWEEP_FRAUD               8 

# KESIMPULAN

## GROCERY — volume terbesar tapi loss_rate rendah
GROCERY SWEEP_FRAUD  → loss $1,212/tahun, loss_rate 0.06% 🟢

GROCERY VOID_ABUSE   → loss $806/tahun,  loss_rate 0.04% 🟢

Revenue GROCERY = $4,093,814 → terbesar!

Loss rate sangat kecil karena revenue base-nya besar.

## COSMETICS — CRITICAL meski nominal kecil

COSMETICS VOID_ABUSE → loss $112/tahun  🔴 CRITICAL
                     → loss_rate 0.69%!

Revenue COSMETICS = $32,360 → sangat kecil

Loss $224 dari revenue $32,360 = proporsi SANGAT tinggi!

## DRUG GM — WARNING di dua tipe shrinkage

DRUG GM VOID_ABUSE  → $921/tahun  🟡

DRUG GM SWEEP_FRAUD → $870/tahun  🟡

Total DRUG GM       → $1,791/tahun ← kedua tertinggi setelah GROCERY!

BQ1: Kategori paling rentan?
* → DRUG GM (volume) & COSMETICS (loss rate tertinggi 0.69%)
* → Produk: Candy, Magazine, Hair Care, Makeup

BQ2: Pola waktu anomali?
* → Midnight 00-06: avg fraud value $26 (sweep fraud)
* → Early Morning 06-09: fraud rate 0.63% (shift change)

BQ3: Store dengan anomali rate abnormal?
* → STORE 293: fraud_rate 0.32%, loss $423  🔴
* → STORE 3001, 569, 3182: HIGH RISK        🔴

BQ4: Estimasi kerugian tahunan?
* → COSMETICS: CRITICAL priority (loss_rate 0.69%)
* → DRUG GM: WARNING (loss $1,791/tahun)
* → Top tipe: SWEEP_FRAUD & VOID_ABUSE

# Save Query Result

In [16]:
# Simpan semua hasil query ke Excel untuk Power BI nanti
import os
os.makedirs('outputs', exist_ok=True)

with pd.ExcelWriter('outputs/sql_analysis_results.xlsx') as writer:
    q1.to_excel(writer, sheet_name='Q1_Product_Vulnerability', index=False)
    q2.to_excel(writer, sheet_name='Q2_Time_Pattern',          index=False)
    q3_fixed.to_excel(writer, sheet_name='Q3_Store_Risk',      index=False)
    q4.to_excel(writer, sheet_name='Q4_Annual_Loss',           index=False)

print("✅ Hasil SQL disimpan → outputs/sql_analysis_results.xlsx")

# Tutup koneksi database
conn.close()
print("✅ Database connection closed")

✅ Hasil SQL disimpan → outputs/sql_analysis_results.xlsx
✅ Database connection closed
